# Chapter 27 — Time Series & Autocorrelation
*Tier 3: Engineers & Data Scientists*

## 🎯 Learning Objectives
- Identify stationarity, trend, and seasonality
- Compute ACF and PACF for model identification
- Fit ARIMA and decompose time series

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose
import pandas as pd

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — Time Series Components

Any time series can be decomposed:
$$Y_t = T_t + S_t + R_t \quad \text{(additive)}$$
$$Y_t = T_t \times S_t \times R_t \quad \text{(multiplicative)}$$

- $T_t$: **Trend** — long-run direction
- $S_t$: **Seasonality** — periodic patterns
- $R_t$: **Residual** — irregular random fluctuations

A series is **stationary** if mean, variance, and autocovariance don't change with time.

In [ ]:
# Generate a time series with trend + seasonality + noise
n = 240  # 20 years of monthly data
t = np.arange(n)
trend     = 0.05 * t
seasonal  = 10 * np.sin(2*np.pi*t/12)
noise_ts  = rng.normal(0, 2, n)
y_ts = 50 + trend + seasonal + noise_ts

dates = pd.date_range("2000-01-01", periods=n, freq="ME")
ts = pd.Series(y_ts, index=dates)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0,0].plot(ts); axes[0,0].set_title("Original Series")
axes[0,1].plot(trend); axes[0,1].set_title("Trend Component")
axes[1,0].plot(seasonal[:48]); axes[1,0].set_title("Seasonal Component (4 cycles)")
axes[1,1].plot(noise_ts[:60]); axes[1,1].set_title("Residuals (first 60)")
plt.suptitle("Time Series Decomposition Components", fontweight="bold")
plt.tight_layout(); plt.show()

## 2. Math Walkthrough — ACF and PACF

**Autocorrelation at lag $k$:**
$$\rho_k = \frac{\text{Cov}(Y_t, Y_{t-k})}{\text{Var}(Y_t)}$$

**ARIMA(p, d, q):**
- **p**: AR order (PACF cuts off after lag p)
- **d**: differencing order (to achieve stationarity)
- **q**: MA order (ACF cuts off after lag q)

In [ ]:
# Stationarity test and differencing
result = adfuller(y_ts, autolag="AIC")
print(f"ADF test on original series: stat={result[0]:.3f}, p={result[1]:.4f}")
print("Stationary ✅" if result[1] < 0.05 else "Non-stationary — need differencing")

y_diff = np.diff(y_ts)
result_d = adfuller(y_diff, autolag="AIC")
print(f"
ADF test after 1st differencing: stat={result_d[0]:.3f}, p={result_d[1]:.4f}")
print("Stationary ✅" if result_d[1] < 0.05 else "Still non-stationary")

In [ ]:
# Decomposition and ARIMA modelling
decomp = seasonal_decompose(ts, model="additive", period=12)
fig = decomp.plot()
fig.set_size_inches(12, 8)
plt.suptitle("STL Decomposition", fontweight="bold")
plt.tight_layout(); plt.show()

# Fit ARIMA(1,1,1) with seasonal component
model_arima = ARIMA(ts, order=(1,1,1), seasonal_order=(1,1,1,12)).fit()
forecast = model_arima.forecast(steps=24)
plt.figure(figsize=(12, 5))
ts[-48:].plot(label="Observed")
forecast.plot(label="Forecast (24 months)", style="r--")
plt.title("SARIMA(1,1,1)(1,1,1)[12] Forecast")
plt.legend(); plt.tight_layout(); plt.show()
print(f"
AIC: {model_arima.aic:.1f}")